# CPE15 - Week 3: NumPy Matrix Operations and Linear Algebra

**Programming for Data Science | Professional Elective 1 | AY 2026-2027**

| Syllabus element | Alignment |
|---|---|
| Course outcome | CO2 |
| Course objective | COBJ1 |
| Assessment connection | NumPy numerical performance output |
| Sustainable Development Goals | 4, 9 |

This notebook is designed for explanation, live coding, guided practice, and independent follow-through. Run it from top to bottom in a fresh kernel so that every result can be reproduced.


## Lecture map

1. Matrix multiplication is a composition of relationships
2. Transpose, determinant, rank, and condition number
3. Solving a linear system: nodal-voltage example
4. Eigenvalues and eigenvectors
5. Performance-output pattern

            **Live-teaching rhythm:** define the idea → predict the result → run a focused example → inspect the saved output → explain the evidence → complete the practice task.

## Learning outcomes
   By the end of the session, students should be able to:

- distinguish elementwise multiplication from matrix multiplication;
- compute and interpret transpose, determinant, rank, and condition number;
- solve a system of linear equations with `numpy.linalg.solve`;
- interpret eigenvalues and eigenvectors in a small engineering example;
- recognize when inverse-based computation is unnecessary or unstable.

            ## Lecture route

            1. Matrix shape and multiplication
2. Transpose, determinant, rank, and inverse
3. Solving simultaneous equations
4. Eigenvalues and eigenvectors
5. Numerical checks and interpretation


In [ ]:
from pathlib import Path
import platform
import random
import sys

random.seed(15)
ARTIFACTS = Path("artifacts")
ARTIFACTS.mkdir(exist_ok=True)

print(f"Python: {sys.version.split()[0]}")
print(f"Platform: {platform.system()}")
print(f"Artifacts folder: {ARTIFACTS.resolve()}")


Python: 3.13.7
Platform: Windows
Artifacts folder: C:\Users\Ranj\Documents\SLSU\CEN\CPE15\2026-2027 Lecture Notebooks\artifacts


> **Reproducibility habit:** a notebook is not finished merely because it ran once. It should run in order from a restarted kernel, use explicit inputs, avoid hidden state, and explain the meaning of its outputs.


## 1. Matrix multiplication is a composition of relationships

For `A @ B`, the inner dimensions must match. If `A` has shape `(m, n)` and `B` has shape `(n, p)`, the result has shape `(m, p)`. Each output entry is a dot product between one row of `A` and one column of `B`.

Elementwise `A * B` answers a different question and requires equal or broadcast-compatible shapes.


In [ ]:
import numpy as np

A = np.array([[1.0, 2.0], [3.0, 4.0]])
B = np.array([[2.0, 0.0], [1.0, 2.0]])

print("Elementwise A * B:", A * B, sep=chr(10))
print("Matrix product A @ B:", A @ B, sep=chr(10))
print("Dot product of first row and first column:", A[0] @ B[:, 0])


Elementwise A * B:
[[2. 0.]
 [3. 8.]]
Matrix product A @ B:
[[ 4.  4.]
 [10.  8.]]
Dot product of first row and first column: 4.0


### Additional worked case — matrix multiplication requires compatible inner dimensions

Elementwise multiplication combines values at matching positions. Matrix multiplication composes row-to-column relationships: an array with shape `(m, n)` may multiply one with shape `(n, p)`, producing `(m, p)`. The shared inner dimension represents the features being combined. Checking shapes before `@` prevents treating an accidental numerical result as a meaningful model.

In [ ]:
clinic_design = np.array([[1.0, 2.0, 3.0], [4.0, 5.0, 6.0]])
clinic_weights = np.array([[0.5], [1.0], [-0.5]])
clinic_score = clinic_design @ clinic_weights

print("Shapes:", clinic_design.shape, "@", clinic_weights.shape, "=", clinic_score.shape)
print("Scores:", clinic_score.ravel())
print("First score checked manually:", 1 * 0.5 + 2 * 1.0 + 3 * -0.5)

Shapes: (2, 3) @ (3, 1) = (2, 1)
Scores: [1. 4.]
First score checked manually: 1.0


**Result and interpretation.** The `(2, 3) @ (3, 1)` product returns two scores. The manual dot-product check reproduces the first score and makes the row-by-column calculation visible.

### Science communication lens

- **Audience:** An engineering reader who must connect the matrix result to the encoded physical or statistical system.
- **Lead with the meaning:** The `(2, 3) @ (3, 1)` product returns two scores.
- **Show the evidence:** Define each matrix and vector, state dimensions and units, report the solution or diagnostic, and show residual, identity, eigenpair, or conditioning checks.
- **State the boundary:** Numerical verification shows that the encoded equations were solved; it does not prove that those equations or measurements represent reality adequately.

An effective explanation follows **claim → evidence → reasoning → limitation**. Define technical terms when first used, retain units and denominators, distinguish observed output from interpretation, and use wording proportional to the strength of the evidence.

### Practice task — your turn

**Task.** Multiply a 2 × 3 measurement matrix by a 3 × 2 transformation matrix, annotate the meaning of every dimension, and verify one output element manually.

**Before running your solution.** Predict the most relevant result property: value, type, shape, retained row count, numerical range, visual pattern, map behavior, or artifact destination.

**After running your solution.** Compare the output with your prediction. Report the evidence with applicable units and counts, explain the reasoning, and state one assumption or limitation.

In [ ]:
# YOUR TURN — Matrix multiplication is a composition of relationships
# Complete the specific task in the Markdown cell above.
# Keep input values, intermediate results, and final evidence easy to inspect.

# Write your solution below.

## 2. Transpose, determinant, rank, and condition number

The transpose exchanges rows and columns. The determinant indicates whether a square matrix is singular: a determinant of zero means no inverse exists. Rank describes the number of linearly independent directions.

The condition number is a practical warning about sensitivity. A very large value means small input changes may produce large solution changes. Do not treat a nonzero determinant alone as proof that a numerical result is trustworthy.


In [ ]:
M = np.array([[4.0, 2.0], [1.0, 3.0]])

diagnostics = {
    "transpose": M.T,
    "determinant": float(np.linalg.det(M)),
    "rank": int(np.linalg.matrix_rank(M)),
    "condition_number": float(np.linalg.cond(M)),
}

diagnostics

{'transpose': array([[4., 1.],
        [2., 3.]]),
 'determinant': 10.000000000000002,
 'rank': 2,
 'condition_number': 2.618033988749895}

### Inverse and identity verification

The diagnostics indicate whether an inverse is numerically plausible. This continuation computes it and verifies the defining relationship `M @ inv(M) ≈ I`. `np.allclose` is used because floating-point arithmetic may leave tiny rounding differences. For solving `Mx=b`, prefer `np.linalg.solve` rather than explicitly multiplying by the inverse.

In [ ]:
M_inverse = np.linalg.inv(M)
identity_check = M @ M_inverse
print("Inverse:", np.round(M_inverse, 3), sep=chr(10))
print("M @ inv(M):", np.round(identity_check, 10), sep=chr(10))
assert np.allclose(identity_check, np.eye(2))


Inverse:
[[ 0.3 -0.2]
 [-0.1  0.4]]
M @ inv(M):
[[ 1.  0.]
 [-0.  1.]]


Computing an inverse is useful for teaching, but it is usually better to solve `Ax = b` directly with `solve(A, b)`. Direct solving is clearer, more efficient, and generally more numerically stable than writing `inv(A) @ b`.


### Discussion clinic — each matrix diagnostic answers a different question

The transpose swaps axes; it does not invert a matrix. The determinant is a scale-and-orientation diagnostic and equals zero for a singular square matrix. Rank counts independent directions. The condition number estimates how strongly input error may be amplified in a solve. A matrix may be technically invertible yet so ill-conditioned that a solution is unreliable. Diagnostics should be interpreted together and in relation to measurement precision.

**Interpretation standard.** Do not report a determinant or condition number without a conclusion. Explain whether the matrix is singular, nearly dependent, or acceptably stable for the intended numerical accuracy.

### 2.1 Individual topic — Transpose

**What it is and how it works.** The transpose swaps matrix axes, turning rows into columns. It changes shape but not the stored scalar values.

**Core syntax**

```python
`matrix.T`; `np.transpose(matrix)`
```

**When to use it.** Use it to align dimensions, switch observation/feature orientation, or express linear-algebra definitions.

**When to use another approach.** Do not call transpose an inverse; it does not undo a general matrix transformation.

In [ ]:
# Demonstration — Transpose
topic_matrix = np.array([[1, 2, 3], [4, 5, 6]])
print("original shape:", topic_matrix.shape)
print("transpose shape:", topic_matrix.T.shape)
print(topic_matrix.T, sep=chr(10))

original shape: (2, 3)
transpose shape: (3, 2)
[[1 4]
 [2 5]
 [3 6]]


**Expected output pattern and interpretation.** A `(2, 3)` matrix becomes `(3, 2)`, with the first original row becoming the first values across the transposed columns.

**Science-communication statement.** State the original and new axis meanings; transposition changes representation, not the underlying observations.

### 2.2 Individual topic — Determinant

**What it is and how it works.** The determinant is a scalar defined for square matrices. Zero indicates singularity; magnitude relates to volume scaling and sign to orientation.

**Core syntax**

```python
`np.linalg.det(matrix)`
```

**When to use it.** Use it as one diagnostic for invertibility and transformation scaling in small teaching systems.

**When to use another approach.** Avoid using determinant alone to judge numerical stability; condition number is more informative for sensitivity.

In [ ]:
# Demonstration — Determinant
topic_nonsingular = np.array([[2.0, 0.0], [0.0, 3.0]])
topic_singular = np.array([[1.0, 2.0], [2.0, 4.0]])
print("det nonsingular:", np.linalg.det(topic_nonsingular))
print("det singular:", np.linalg.det(topic_singular))

det nonsingular: 6.0
det singular: 0.0


**Expected output pattern and interpretation.** The diagonal matrix has determinant 6, while the dependent-row matrix has determinant 0 and cannot be inverted uniquely.

**Science-communication statement.** Report determinant with matrix scale and precision; use 'numerically near zero' rather than overclaiming exact singularity for measured coefficients.

### 2.3 Individual topic — Rank

**What it is and how it works.** Matrix rank counts linearly independent rows or columns and indicates how many independent directions the matrix represents.

**Core syntax**

```python
`np.linalg.matrix_rank(matrix)`
```

**When to use it.** Use it to diagnose redundant equations or features and determine whether a system has full rank.

**When to use another approach.** Avoid interpreting numerical rank without considering tolerance and measurement precision.

In [ ]:
# Demonstration — Rank
topic_full_rank = np.array([[1.0, 0.0], [0.0, 1.0]])
topic_low_rank = np.array([[1.0, 2.0], [2.0, 4.0]])
print("full rank:", np.linalg.matrix_rank(topic_full_rank))
print("dependent rank:", np.linalg.matrix_rank(topic_low_rank))

full rank: 2
dependent rank: 1


**Expected output pattern and interpretation.** The identity matrix has rank 2; the matrix with one row twice the other has rank 1.

**Science-communication statement.** Say that one encoded direction is redundant under the numerical tolerance; do not infer why without domain evidence.

### 2.4 Individual topic — Condition number

**What it is and how it works.** The condition number estimates worst-case amplification of relative input error in a numerical problem. Larger values indicate greater sensitivity.

**Core syntax**

```python
`np.linalg.cond(matrix)`
```

**When to use it.** Use it before trusting a solve or inverse based on measured or rounded coefficients.

**When to use another approach.** Do not use one universal cutoff without considering dtype, precision, scale, and required accuracy.

In [ ]:
# Demonstration — Condition number
topic_stable = np.eye(2)
topic_sensitive = np.array([[1.0, 1.0], [1.0, 1.000001]])
print("identity condition:", np.linalg.cond(topic_stable))
print("sensitive condition:", f"{np.linalg.cond(topic_sensitive):.2e}")

identity condition: 1.0
sensitive condition: 4.00e+06


**Expected output pattern and interpretation.** The identity has condition number 1, while the nearly dependent matrix has a very large condition number.

**Science-communication statement.** Describe the second system as sensitive to small input changes; do not state that every computed value is automatically useless.

### 2.5 Individual topic — Inverse

**What it is and how it works.** For an invertible square matrix, the inverse satisfies `A @ A⁻¹ = I`. Numerical inversion is more expensive and often less appropriate than a direct solve.

**Core syntax**

```python
`inverse = np.linalg.inv(A)`; `np.allclose(A @ inverse, np.eye(n))`
```

**When to use it.** Use it when the inverse itself has a justified interpretation or for teaching matrix properties.

**When to use another approach.** Prefer `np.linalg.solve(A, b)` when the actual goal is solving `Ax=b`.

In [ ]:
# Demonstration — Inverse
topic_A = np.array([[4.0, 1.0], [2.0, 3.0]])
topic_inverse = np.linalg.inv(topic_A)
print(topic_inverse, sep=chr(10))
print("identity check:", np.allclose(topic_A @ topic_inverse, np.eye(2)))

[[ 0.3 -0.1]
 [-0.2  0.4]]
identity check: True


**Expected output pattern and interpretation.** The displayed inverse passes the approximate identity check, allowing for floating-point rounding.

**Science-communication statement.** Call this a numerical verification of the inverse relationship; it does not validate the model represented by `A`.

### Science communication lens

- **Audience:** An engineering reader who must connect the matrix result to the encoded physical or statistical system.
- **Lead with the meaning:** Do not report a determinant or condition number without a conclusion.
- **Show the evidence:** Define each matrix and vector, state dimensions and units, report the solution or diagnostic, and show residual, identity, eigenpair, or conditioning checks.
- **State the boundary:** Numerical verification shows that the encoded equations were solved; it does not prove that those equations or measurements represent reality adequately.

An effective explanation follows **claim → evidence → reasoning → limitation**. Define technical terms when first used, retain units and denominators, distinguish observed output from interpretation, and use wording proportional to the strength of the evidence.

### Practice task — your turn

**Task.** For one well-scaled matrix and one nearly singular matrix, calculate determinant, rank, and condition number and explain why these diagnostics should be interpreted together.

**Before running your solution.** Predict the most relevant result property: value, type, shape, retained row count, numerical range, visual pattern, map behavior, or artifact destination.

**After running your solution.** Compare the output with your prediction. Report the evidence with applicable units and counts, explain the reasoning, and state one assumption or limitation.

In [ ]:
# YOUR TURN — Transpose, determinant, rank, and condition number
# Complete the specific task in the Markdown cell above.
# Keep input values, intermediate results, and final evidence easy to inspect.

# Write your solution below.

## 3. Solving a linear system: nodal-voltage example

Linear systems appear in circuit analysis, calibration, force balance, and many estimation problems. The coefficient matrix represents the relationships among unknowns; the right-hand side represents known inputs.

The example solves two nodal equations. After obtaining the solution, always substitute it back and inspect the residual `A @ x - b`.


In [ ]:
conductance_matrix = np.array(
    [
        [0.30, -0.10],
        [-0.10, 0.25],
    ]
)
injected_current_a = np.array([1.2, 0.5])

node_voltage_v = np.linalg.solve(conductance_matrix, injected_current_a)
residual = conductance_matrix @ node_voltage_v - injected_current_a

print("Node voltages (V):", np.round(node_voltage_v, 4))
print("Residual:", residual)
assert np.allclose(residual, 0)


Node voltages (V): [5.3846 4.1538]
Residual: [0. 0.]


### What if the system is singular?

A singular matrix may represent redundant equations or insufficient independent information. Catching the exception is better than allowing a report to continue with a nonexistent solution.


In [ ]:
singular = np.array([[1.0, 2.0], [2.0, 4.0]])
target = np.array([3.0, 6.0])

try:
    np.linalg.solve(singular, target)
except np.linalg.LinAlgError as error:
    print(f"Cannot compute a unique solution: {error}")


Cannot compute a unique solution: Singular matrix


### Discussion clinic — solve the system directly and verify the residual

`np.linalg.solve(A, b)` solves `Ax = b` without explicitly forming `A⁻¹`, which is generally clearer and more numerically appropriate. Units matter: a conductance matrix multiplied by voltage produces current. After solving, recompute `A @ x` and subtract `b`. A near-zero residual confirms that the numerical result satisfies the encoded equations, but it cannot validate an incorrect physical model or wrong coefficients.

**Interpretation standard.** State the unknown vector, coefficient units, solution units, residual magnitude, and condition number. These make a linear-system result auditable rather than merely printable.

### 3.1 Individual topic — `np.linalg.solve`

**What it is and how it works.** `solve(A, b)` finds `x` satisfying the square full-rank system `Ax=b` without explicitly computing an inverse.

**Core syntax**

```python
`x = np.linalg.solve(A, b)`
```

**When to use it.** Use it for one or several right-hand sides when the coefficient matrix is square and nonsingular.

**When to use another approach.** Use least squares or another appropriate method for overdetermined, underdetermined, or rank-deficient systems.

In [ ]:
# Demonstration — `np.linalg.solve`
topic_A_solve = np.array([[3.0, 1.0], [1.0, 2.0]])
topic_b_solve = np.array([9.0, 8.0])
topic_x_solve = np.linalg.solve(topic_A_solve, topic_b_solve)
print("solution:", topic_x_solve)

solution: [2. 3.]


**Expected output pattern and interpretation.** The solution is `[2, 3]` because the two equations are satisfied simultaneously.

**Science-communication statement.** Define unknowns and units before reporting `[2, 3]`; an unlabeled vector is not a scientific result.

### 3.2 Individual topic — Residual and `np.allclose`

**What it is and how it works.** The residual `A @ x - b` measures how closely a computed solution satisfies the encoded equations. `allclose` compares within floating tolerances.

**Core syntax**

```python
`residual = A @ x - b`; `np.allclose(residual, 0)`
```

**When to use it.** Use residual checks after every numerical solve and report a magnitude appropriate to units.

**When to use another approach.** Do not treat a small residual as proof that coefficients, assumptions, or data are correct.

In [ ]:
# Demonstration — Residual and `np.allclose`
topic_residual = topic_A_solve @ topic_x_solve - topic_b_solve
print("residual:", topic_residual)
print("maximum absolute residual:", np.abs(topic_residual).max())
print("close to zero:", np.allclose(topic_residual, 0))

residual: [0. 0.]
maximum absolute residual: 0.0
close to zero: True


**Expected output pattern and interpretation.** The residual is zero within floating precision, confirming that the numerical solution satisfies the two supplied equations.

**Science-communication statement.** Report residual with its unit and pair it with condition number and model assumptions.

### Science communication lens

- **Audience:** An engineering reader who must connect the matrix result to the encoded physical or statistical system.
- **Lead with the meaning:** State the unknown vector, coefficient units, solution units, residual magnitude, and condition number.
- **Show the evidence:** Define each matrix and vector, state dimensions and units, report the solution or diagnostic, and show residual, identity, eigenpair, or conditioning checks.
- **State the boundary:** Numerical verification shows that the encoded equations were solved; it does not prove that those equations or measurements represent reality adequately.

An effective explanation follows **claim → evidence → reasoning → limitation**. Define technical terms when first used, retain units and denominators, distinguish observed output from interpretation, and use wording proportional to the strength of the evidence.

### Practice task — your turn

**Task.** Solve a three-equation nodal-voltage system with `np.linalg.solve`, calculate the residual vector, and report the maximum absolute residual with units.

**Before running your solution.** Predict the most relevant result property: value, type, shape, retained row count, numerical range, visual pattern, map behavior, or artifact destination.

**After running your solution.** Compare the output with your prediction. Report the evidence with applicable units and counts, explain the reasoning, and state one assumption or limitation.

In [ ]:
# YOUR TURN — Solving a linear system: nodal-voltage example
# Complete the specific task in the Markdown cell above.
# Keep input values, intermediate results, and final evidence easy to inspect.

# Write your solution below.

## 4. Eigenvalues and eigenvectors

For a square matrix `A`, an eigenvector `v` is a nonzero direction whose orientation is preserved by the transformation: `A @ v = lambda * v`. The scalar `lambda` is its eigenvalue.

In a covariance matrix, eigenvectors identify orthogonal directions of variation and eigenvalues quantify the variation along those directions. This is the foundation of principal component analysis, though this week focuses on the linear-algebra meaning.


In [ ]:
covariance = np.array([[4.0, 1.8], [1.8, 1.5]])
eigenvalues, eigenvectors = np.linalg.eigh(covariance)
order = np.argsort(eigenvalues)[::-1]
eigenvalues = eigenvalues[order]
eigenvectors = eigenvectors[:, order]

print("Eigenvalues:", np.round(eigenvalues, 4))
print("Eigenvectors by column:", np.round(eigenvectors, 4), sep=chr(10))
print("Explained share:", np.round(eigenvalues / eigenvalues.sum(), 4))


Eigenvalues: [4.9415 0.5585]
Eigenvectors by column:
[[-0.8861  0.4635]
 [-0.4635 -0.8861]]
Explained share: [0.8984 0.1016]


### Verify the first eigenpair

The previous cell produced sorted eigenvalues and eigenvectors. This continuation selects the first eigenvector column and checks the defining equation `A @ v = λv`. Matching sides verify the computed pair numerically. The sign of an eigenvector may reverse while representing the same direction, so compare the equation rather than memorizing signs.

In [ ]:
first_vector = eigenvectors[:, 0]
left = covariance @ first_vector
right = eigenvalues[0] * first_vector

print("A @ v:", np.round(left, 6))
print("lambda * v:", np.round(right, 6))
assert np.allclose(left, right)


A @ v: [-4.378697 -2.290206]
lambda * v: [-4.378697 -2.290206]


Use `eigh` for real symmetric or complex Hermitian matrices, such as covariance matrices. It exploits the structure and returns real eigenvalues. Eigenvector signs may differ across valid implementations: `v` and `-v` describe the same direction.


### Discussion clinic — an eigenpair must satisfy its defining equation

For an eigenvector `v` and eigenvalue `λ`, the relationship is `A @ v = λ * v`. `np.linalg.eigh` is designed for real symmetric or Hermitian matrices and returns orthonormal eigenvectors. Eigenvector signs are not unique: `v` and `-v` describe the same direction. Sorting eigenvalues changes the column order of the eigenvector matrix, so both must be reordered together.

**Interpretation standard.** Verify at least one eigenpair with `np.allclose(A @ v, λ * v)`. Explained share describes relative variance only when the matrix and application justify that interpretation.

### 4.1 Individual topic — Eigenvalue

**What it is and how it works.** An eigenvalue is the scale factor associated with a direction that a square matrix transforms without changing that direction.

**Core syntax**

```python
`eigenvalues, eigenvectors = np.linalg.eig(A)`
```

**When to use it.** Use eigenvalues for stability, modes, covariance directions, and other problems whose mathematics supports them.

**When to use another approach.** Do not interpret eigenvalues independently of the matrix definition and units.

In [ ]:
# Demonstration — Eigenvalue
topic_diag = np.diag([4.0, 2.0])
topic_eigenvalues, topic_eigenvectors = np.linalg.eig(topic_diag)
print("eigenvalues:", topic_eigenvalues)

eigenvalues: [4. 2.]


**Expected output pattern and interpretation.** The diagonal entries 4 and 2 are the eigenvalues because each coordinate direction is scaled independently.

**Science-communication statement.** State what the matrix represents before attaching scientific meaning to the magnitude of an eigenvalue.

### 4.2 Individual topic — Eigenvector

**What it is and how it works.** An eigenvector is a nonzero direction `v` satisfying `A @ v = λv`. Its length is conventionally normalized, and its sign is not unique.

**Core syntax**

```python
`v = eigenvectors[:, index]`; `np.allclose(A @ v, lambda_value * v)`
```

**When to use it.** Use it to identify invariant directions or modes supported by the matrix.

**When to use another approach.** Do not compare signs across software runs as though sign reversal changed the underlying direction.

In [ ]:
# Demonstration — Eigenvector
topic_v = topic_eigenvectors[:, 0]
topic_lambda = topic_eigenvalues[0]
print("eigenvector:", topic_v)
print("equation holds:", np.allclose(topic_diag @ topic_v, topic_lambda * topic_v))

eigenvector: [1. 0.]
equation holds: True


**Expected output pattern and interpretation.** The selected vector satisfies the defining equation for its eigenvalue.

**Science-communication statement.** Describe the direction and verification; avoid calling it important until a domain-specific criterion establishes importance.

### 4.3 Individual topic — Explained share from covariance eigenvalues

**What it is and how it works.** For a positive-semidefinite covariance matrix, dividing each eigenvalue by their sum gives the share of total variance associated with each eigen-direction.

**Core syntax**

```python
`share = eigenvalues / eigenvalues.sum()`
```

**When to use it.** Use it in PCA-style variance summaries after confirming the matrix is a covariance or related valid matrix.

**When to use another approach.** Do not call variance share 'information retained' without defining information and considering scaling and preprocessing.

In [ ]:
# Demonstration — Explained share from covariance eigenvalues
topic_covariance = np.array([[4.0, 0.0], [0.0, 1.0]])
topic_cov_values = np.linalg.eigvalsh(topic_covariance)[::-1]
topic_share = topic_cov_values / topic_cov_values.sum()
print("eigenvalues:", topic_cov_values)
print("variance shares:", topic_share)

eigenvalues: [4. 1.]
variance shares: [0.8 0.2]


**Expected output pattern and interpretation.** The first direction accounts for 80% and the second for 20% of total variance in this simple covariance matrix.

**Science-communication statement.** Say '80% of variance in this scaled two-variable example,' preserving the matrix context and avoiding a broader importance claim.

### Science communication lens

- **Audience:** An engineering reader who must connect the matrix result to the encoded physical or statistical system.
- **Lead with the meaning:** Verify at least one eigenpair with `np.allclose(A @ v, λ * v)`.
- **Show the evidence:** Define each matrix and vector, state dimensions and units, report the solution or diagnostic, and show residual, identity, eigenpair, or conditioning checks.
- **State the boundary:** Numerical verification shows that the encoded equations were solved; it does not prove that those equations or measurements represent reality adequately.

An effective explanation follows **claim → evidence → reasoning → limitation**. Define technical terms when first used, retain units and denominators, distinguish observed output from interpretation, and use wording proportional to the strength of the evidence.

### Practice task — your turn

**Task.** Compute the eigenpairs of a 2 × 2 matrix, select one pair, and verify numerically that `A @ v` and `λ * v` agree within floating-point tolerance.

**Before running your solution.** Predict the most relevant result property: value, type, shape, retained row count, numerical range, visual pattern, map behavior, or artifact destination.

**After running your solution.** Compare the output with your prediction. Report the evidence with applicable units and counts, explain the reasoning, and state one assumption or limitation.

In [ ]:
# YOUR TURN — Eigenvalues and eigenvectors
# Complete the specific task in the Markdown cell above.
# Keep input values, intermediate results, and final evidence easy to inspect.

# Write your solution below.

## 5. Performance-output pattern

A strong numerical submission contains the model, dimensions, units, computation, a numerical verification, and an interpretation. Printing a matrix without explaining it is incomplete.


In [ ]:
calibration_matrix = np.array(
    [
        [1.00, 0.02, 0.00],
        [0.01, 0.98, 0.03],
        [0.00, 0.02, 1.01],
    ]
)
measured = np.array([10.2, 4.9, 7.1])
corrected = np.linalg.solve(calibration_matrix, measured)
reconstruction = calibration_matrix @ corrected

assert corrected.shape == (3,)
assert np.allclose(reconstruction, measured)
print("Corrected signal:", np.round(corrected, 4))
print("Maximum reconstruction error:", np.abs(reconstruction - measured).max())


Corrected signal: [10.1063  4.6845  6.9369]
Maximum reconstruction error: 0.0


### Additional worked case — compare a stable reconstruction with a sensitive system

A performance output should include both a computed solution and evidence that the solution reconstructs the measured vector. Sensitivity is a separate issue: a small residual can coexist with an unstable model when the matrix is ill-conditioned. The contrast below perturbs the right-hand side of a nearly dependent system and compares how much the solution changes.

In [ ]:
clinic_sensitive_A = np.array([[1.0, 1.0], [1.0, 1.000001]])
clinic_b1 = np.array([2.0, 2.000001])
clinic_b2 = clinic_b1 + np.array([0.0, 0.000001])
clinic_x1 = np.linalg.solve(clinic_sensitive_A, clinic_b1)
clinic_x2 = np.linalg.solve(clinic_sensitive_A, clinic_b2)

print("Condition number:", f"{np.linalg.cond(clinic_sensitive_A):.2e}")
print("Solution before perturbation:", clinic_x1)
print("Solution after perturbation:", clinic_x2)
print("Solution change:", clinic_x2 - clinic_x1)

Condition number: 4.00e+06
Solution before perturbation: [1. 1.]
Solution after perturbation: [-4.4408921e-10  2.0000000e+00]
Solution change: [-1.  1.]


**Result and interpretation.** The large condition number warns that a tiny measurement change can cause a much larger solution change. This is why reconstruction, residual, and conditioning belong in the same performance report.

### Science communication lens

- **Audience:** An engineering reader who must connect the matrix result to the encoded physical or statistical system.
- **Lead with the meaning:** The large condition number warns that a tiny measurement change can cause a much larger solution change.
- **Show the evidence:** Define each matrix and vector, state dimensions and units, report the solution or diagnostic, and show residual, identity, eigenpair, or conditioning checks.
- **State the boundary:** Numerical verification shows that the encoded equations were solved; it does not prove that those equations or measurements represent reality adequately.

An effective explanation follows **claim → evidence → reasoning → limitation**. Define technical terms when first used, retain units and denominators, distinguish observed output from interpretation, and use wording proportional to the strength of the evidence.

### Practice task — your turn

**Task.** Time a loop and a vectorized calculation on the same input, confirm equal numerical results, and report median timings without claiming that one timing generalizes to all workloads.

**Before running your solution.** Predict the most relevant result property: value, type, shape, retained row count, numerical range, visual pattern, map behavior, or artifact destination.

**After running your solution.** Compare the output with your prediction. Report the evidence with applicable units and counts, explain the reasoning, and state one assumption or limitation.

In [ ]:
# YOUR TURN — Performance-output pattern
# Complete the specific task in the Markdown cell above.
# Keep input values, intermediate results, and final evidence easy to inspect.

# Write your solution below.

## Common mistakes

- Using `*` when the model requires `@`.
- Multiplying matrices in the wrong order; matrix multiplication is not commutative.
- Computing an inverse when `solve` is the actual goal.
- Reporting a solution without checking residuals or condition number.
- Interpreting eigenvector signs as inherently positive or negative directions.


## Independent challenge

Model and solve a three-variable engineering system. State what each row, column, unknown, and unit represents; report determinant, rank, and condition number; solve without explicitly computing the inverse; verify the residual; and interpret whether the solution appears reliable.

**Submission expectation:** include readable code, meaningful variable names, a short interpretation, and evidence that the notebook was restarted and run from top to bottom.


## Key takeaways

            - Matrix dimensions express whether a multiplication is meaningful.
- A direct linear solve should be verified by substitution and residuals.
- Condition number adds practical information beyond determinant.
- Eigenpairs describe directions preserved by a linear transformation.

            ## Exit ticket

            1. Why is `solve(A, b)` preferred over `inv(A) @ b`?
2. What does a large condition number warn about?
3. How can you verify a computed eigenpair?


## References and further reading

            - NumPy Developers. NumPy Linear Algebra documentation.
- VanderPlas, J. (2022). Python Data Science Handbook (2nd ed.).
- CPE15 syllabus, Week 3 course learning plan.

            <details>
            <summary><strong>Instructor facilitation note</strong></summary>

            Ask students to predict outputs before execution, compare at least two valid approaches, and explain results in plain language. During live coding, deliberately trigger one common error and model a calm debugging process.

            </details>
